In [ ]:
from langgraph.graph import StateGraph, END, START
from langgraph.checkpoint.memory import InMemorySaver
    
from langchain_groq import ChatGroq
from typing import TypedDict
from dotenv import load_dotenv
load_dotenv()

In [ ]:
llm = ChatGroq(model="llama-3.1-8b-instant")

In [ ]:
class JokeState(TypedDict):
    title: str
    joke: str
    explanation: str


In [ ]:
def generate_joke(state: JokeState):
    prompt = f"generate a joke on topic {state['title']}"
    response = llm.invoke(prompt).content

    return{ 'joke': response} 

In [ ]:
def explain_joke(state: JokeState):
    prompt = f"explain a joke on {state['joke']} clearly"

    response = llm.invoke(prompt).content
    return {'explanation':response}

In [ ]:
config1 = {"configurable": {"thread_id": "jpmc1"}}
chekpointer = InMemorySaver()

graph = StateGraph(JokeState)

graph.add_node("generate_joke", generate_joke)
graph.add_node("explain_joke", explain_joke)

graph.add_edge(START, "generate_joke")
graph.add_edge("generate_joke", "explain_joke")
graph.add_edge("explain_joke", END)

initial_state= {'topic' :"pizza"}

workflow = graph.compile(checkpointer = checkpointer)
workflow.invoke({initial_state}, config = config1)